# TAP access to SOHO metadata

To install [astroquery](https://astroquery.readthedocs.io/en/latest/index.html) ([Ginsburg et al, 2019](https://doi.org/10.3847/1538-3881/aafc33)) to be able to use [TAP+](https://astroquery.readthedocs.io/en/latest/utils/tap.html):

`conda install -c astropy astroquery`

The different missions have their own TAP servers; SOHO's is:
`https://ssa.esac.esa.int/ssa-sl-tap/data? `

## Tables: metadata of metadata

First of all, it's helpful to know what tables are there. 

From example 1.1 on [Astroquery TAP page](https://astroquery.readthedocs.io/en/latest/utils/tap.html), load the table objects using the method `load_tables()`, then print the names of those table objects using the `get_qualified_name()` method. This returns the schema (e.g., csa.) and the table names (e.g., csa.dataset). We only need to focus on the `csa.` schema tables.

In [9]:
from astroquery.utils.tap.core import TapPlus

SOHO = TapPlus(url="https://ssa.esac.esa.int/ssa-sl-tap/tap")
tables = SOHO.load_tables()#only_names=True)

for table in tables:
    print(table.get_qualified_name())

for table in tables:
    if table.get_qualified_name().startswith("soho.v_"):
        print(table.get_qualified_name())

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]
public.dual
soho.campaign
soho.cds_observation
soho.celias_observation
soho.coronal_image
soho.costep_observation
soho.eit_observation
soho.erne_observation
soho.full_disk_solar_image
soho.golf_observation
soho.instrument
soho.lasco_observation
soho.lyra_observation
soho.mdi_observation
soho.mv_observation
soho.observation
soho.observatory
soho.observing_mode
soho.science_object
soho.science_slit
soho.study
soho.sumer_observation
soho.swan_observation
soho.swap_observation
soho.uvcs_observation
soho.virgo_observation
tap_config.coord_sys
tap_config.properties
tap_schema.columns
tap_schema.key_columns
tap_schema.keys
tap_schema.schemas
tap_schema.tables


From the above list of tables, the helpful ones are those beginning `soho.` and the ones meant for users begin with `soho.v_`, where the 'v' means 'view'. The view tables contain the same information as the other `soho.` tables but the columns in those tables might be grouped slightly differently, and the columns containing information meant for internal database use have been taken out since they are of no use to the end-user.

Also from [Astroquery TAP page](https://astroquery.readthedocs.io/en/latest/utils/tap.html), load a specific table and print out the column names. 

In [10]:
table = SOHO.load_table('soho.observation')

print('\n Table: \n', table, '\n')

print('Column names: ')
for column in (table.columns):
    print(column.name)


 Table: 
 TAP Table name: soho.observation
Description: None
Size (bytes): 5867741184
Num. columns: 17 

Column names: 
begin_date
calibrated
campaign_oid
end_date
file_format
file_name
file_path
file_size
observation_oid
observation_type
observing_mode_oid
processing_level
science_objective
science_object_oid
science_slit_oid
study_oid
wavelength_range


## Querying Tables - accessing metadata

So far we have only been loading metadata of the metadata, so we need to make an actual query to get the values:

From [AppDynamics](https://docs.appdynamics.com/display/PRO45/ADQL+Queries) rather than [Astronomical Data Query Language](https://www.ivoa.net/documents/ADQL/2.0), but still usefully put:

`SELECT * | {[DISTINCT] expression [AS alias] [, expression [AS alias]]...}
FROM event_type
[WHERE condition_expression]
[SINCE timevalue [UNTIL timevalue]]
[HAVING condition_expression]
[ORDER BY {field_name | alias} [ASC | DESC] [, {field_name | alias} [ASC | DESC]]...]
[LIMIT integer [, integer]...]`

While the SELECT and FROM clauses are required, all other clauses are optional. Type spaces around each keyword. 

### Whole table

To see all the columns and first 2000 rows (TAP+ has a synchronous request limit of 2000) of a table:

In [12]:
results = SOHO.launch_job("SELECT * FROM soho.observation")
r = results.get_results()
r

begin_date,calibrated,campaign_oid,end_date,file_format,file_name,file_path,file_size,observation_oid,observation_type,observing_mode_oid,processing_level,science_objective,science_object_oid,science_slit_oid,study_oid,wavelength_range
object,bool,int32,object,str20,object,str80,int64,int32,object,int32,str200,object,int32,int32,int32,object
2021-02-19T00:00:00.0,True,--,2021-02-19T23:59:59.0,ASCII,epi21050.pl2,/ssa/sps/science/costep/ephin/2021,13816445,194389001,Pulse Height Analysis (PHA),7,L1_Calibrated,,65,--,--,
2021-03-14T00:00:00.0,False,--,2021-03-14T23:59:59.0,ASCII,epi21073.kor,/ssa/sps/science/costep/ephin/2021,161633,194389003,Correction factors,7,L0_RawData,,65,--,--,
2021-02-26T00:00:00.0,False,--,2021-02-26T23:59:59.0,ASCII,epi21057.kor,/ssa/sps/science/costep/ephin/2021,162311,194389004,Correction factors,7,L0_RawData,,65,--,--,
2021-02-26T00:00:00.0,True,--,2021-02-26T23:59:59.0,ASCII,epi21057.rl2,/ssa/sps/science/costep/ephin/2021,369643,194389005,"Particle Fluxes: e-, H+, He",7,L1_Calibrated,,65,--,--,
2021-03-07T00:00:00.0,True,--,2021-03-07T23:59:59.0,ASCII,epi21066.rl2,/ssa/sps/science/costep/ephin/2021,369897,194389008,"Particle Fluxes: e-, H+, He",7,L1_Calibrated,,65,--,--,
2021-03-07T00:00:00.0,False,--,2021-03-07T23:59:59.0,ASCII,epi21066.kor,/ssa/sps/science/costep/ephin/2021,162311,194389009,Correction factors,7,L0_RawData,,65,--,--,
2021-02-28T00:00:00.0,True,--,2021-02-28T23:59:59.0,ASCII,epi21059.pl2,/ssa/sps/science/costep/ephin/2021,13912921,194389011,Pulse Height Analysis (PHA),7,L1_Calibrated,,65,--,--,
2021-03-02T00:00:00.0,True,--,2021-03-02T23:59:59.0,ASCII,epi21061.pl2,/ssa/sps/science/costep/ephin/2021,13854102,194389013,Pulse Height Analysis (PHA),7,L1_Calibrated,,65,--,--,
2021-03-04T00:00:00.0,True,--,2021-03-04T23:59:59.0,ASCII,epi21063.pl2,/ssa/sps/science/costep/ephin/2021,13835728,194389015,Pulse Height Analysis (PHA),7,L1_Calibrated,,65,--,--,


### Using WHERE to restrict results

To get a list of all datasets that include FGM, we need to query the v_dataset table for the dataset_id and add the WHERE statement to restrict the results to those that contain 'FGM'. Remember that the percentage sign (%) is the wildcard, but it doesn't need to be URL encoded here.

In [16]:
results = SOHO.launch_job("SELECT * FROM soho.observation WHERE processing_level like 'L0_%'")
r = results.get_results()
r

begin_date,calibrated,campaign_oid,end_date,file_format,file_name,file_path,file_size,observation_oid,observation_type,observing_mode_oid,processing_level,science_objective,science_object_oid,science_slit_oid,study_oid,wavelength_range
object,bool,int32,object,str20,object,str80,int64,int32,object,int32,str200,object,int32,int32,int32,object
2021-03-14T00:00:00.0,False,--,2021-03-14T23:59:59.0,ASCII,epi21073.kor,/ssa/sps/science/costep/ephin/2021,161633,194389003,Correction factors,7,L0_RawData,,65,--,--,
2021-02-26T00:00:00.0,False,--,2021-02-26T23:59:59.0,ASCII,epi21057.kor,/ssa/sps/science/costep/ephin/2021,162311,194389004,Correction factors,7,L0_RawData,,65,--,--,
2021-03-07T00:00:00.0,False,--,2021-03-07T23:59:59.0,ASCII,epi21066.kor,/ssa/sps/science/costep/ephin/2021,162311,194389009,Correction factors,7,L0_RawData,,65,--,--,
2021-02-22T00:00:00.0,False,--,2021-02-22T23:59:59.0,ASCII,epi21053.kor,/ssa/sps/science/costep/ephin/2021,162311,194389035,Correction factors,7,L0_RawData,,65,--,--,
2021-03-16T00:00:00.0,False,--,2021-03-16T23:59:59.0,ASCII,epi21075.kor,/ssa/sps/science/costep/ephin/2021,162311,194389043,Correction factors,7,L0_RawData,,65,--,--,
2021-03-09T00:00:00.0,False,--,2021-03-09T23:59:59.0,ASCII,epi21068.kor,/ssa/sps/science/costep/ephin/2021,161860,194389047,Correction factors,7,L0_RawData,,65,--,--,
2021-03-05T00:00:00.0,False,--,2021-03-05T23:59:59.0,ASCII,epi21064.kor,/ssa/sps/science/costep/ephin/2021,162311,194389052,Correction factors,7,L0_RawData,,65,--,--,
2021-02-24T00:00:00.0,False,--,2021-02-24T23:59:59.0,ASCII,epi21055.kor,/ssa/sps/science/costep/ephin/2021,162422,194389058,Correction factors,7,L0_RawData,,65,--,--,
2021-03-11T00:00:00.0,False,--,2021-03-11T23:59:59.0,ASCII,epi21070.kor,/ssa/sps/science/costep/ephin/2021,143666,194389074,Correction factors,7,L0_RawData,,65,--,--,
